In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
from lib.Forecast.SimpleTransformerForecast import SimpleTransformerForecast
from lib.Forecast.PatchTransformerForecast import PatchTransformerForecast, PatchTransformerForecast2
from lib.Forecast.PatchTransformerExoForecast import PatchTransformerExoForecast
from lib.Dataloaders.VirtualAgentDataset import VirtualAgentDataset
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import torch
import numpy as np
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
import os
import time
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Function to train the model
def train_loop(model,
               train,
               val,
               optimizer,
               scheduler=None,
               patience=5,
               epochs=100,
               lossf=F.mse_loss):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    """_Training loop for the model_
    

    Args:
        model: model to train
        optimizer: pytorch optimizer, for example torch.optim.Adam
        train: training data
        val: validation data
        epochs: number of epochs

    Returns:
        _type_: training history, a dictionary with the training and validation loss for each epoch
    """

    def epoch_loss(dataset):
        data_loss = 0.0
        for i, (x_endo, x_exo, x_cond, y_target) in enumerate(dataset):
            # Spostiamo tutto su GPU
            x_endo = x_endo.to(device)
            x_exo = x_exo.to(device)
            x_cond = x_cond.to(device)
            y = y_target.to(device)

            # Passiamo i 3 tensori al modello
            outputs = model(x_endo, exo=x_exo, cond=x_cond).squeeze(-1) # squeeze(-1) rimuove solo l'ultima dim = 1
            loss = lossf(y.squeeze(-1), outputs)
            data_loss += loss.item()
        return data_loss / (i + 1)

    def early_stopping(val_loss, patience=5):
        if len(val_loss) > patience:
            if val_loss[-1] > np.mean(val_loss[-(patience + 1):-1]):
                return True

    hist_loss = {'train': [], 'val': []}
    pbar = tqdm(range(epochs))
    for epoch in pbar:  # loop for all the epochs
        # print(f"Epoch {epoch + 1}/{epochs}")
        model.train()
        for i, (x_endo, x_exo, x_cond, y_target) in enumerate(train):
            x_endo = x_endo.to(device)
            x_exo = x_exo.to(device)
            x_cond = x_cond.to(device)
            y = y_target.to(device)

            optimizer.zero_grad()

            # Reset the gradients
            optimizer.zero_grad()

            # Apply the data to the model
            outputs = model(x_endo, exo=x_exo, cond=x_cond).squeeze(-1)
            # Calculate the loss
            loss = lossf(y.squeeze(-1), outputs)

            # Make the backward pass
            loss.backward()
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        # Calculate the loss in the training and validation sets
        model.eval()
        with torch.no_grad():
            hist_loss['train'].append(
                epoch_loss(train))
            hist_loss['val'].append(
                epoch_loss(val))

        # Show the loss in the training and validation sets
        pbar.set_postfix({
            'train': hist_loss['train'][-1],
            'val': hist_loss['val'][-1],
            'lr': optimizer.param_groups[0]['lr']
        })

        # If the loss in the validation set does not decrease, stop the training
        if early_stopping(hist_loss['val'], patience):
            print("\nEarly stopping reached!")
            break

    return hist_loss

In [ ]:
# --- NUOVA FUNZIONE DI PLOT ---
def plot_prediction(model, data, channel=0):
    # data è la tupla fornita dal VirtualAgentDataset.dataset[idx]
    x_endo, x_exo, x_cond, y_target = data
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Aggiungiamo la dimensione del Batch (unsqueeze(0))
    inputs_endo = x_endo.unsqueeze(0).to(device)
    inputs_exo = x_exo.unsqueeze(0).to(device)
    inputs_cond = x_cond.unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        pred = model(inputs_endo, exo=inputs_exo, cond=inputs_cond).squeeze().cpu().numpy()

    # Prepariamo i dati per il plot
    X_plot = x_endo.cpu().numpy()
    Y_plot = y_target.squeeze().cpu().numpy()

    plt.figure(figsize=(12, 6))
    ax = plt.subplot(1, 2, 1)
    # Disegniamo la storia passata (solo il canale 0, che è il Self)
    plt.plot(X_plot[:, channel], label='Lookback (Distanza VA)', color='blue')
    plt.title('Dato Storico (Passato)')
    plt.legend()
    
    ax = plt.subplot(1, 2, 2)
    plt.plot(Y_plot, label='True Future', color='green', linestyle='dashed')
    plt.plot(pred, label='Predicted Future', color='red')
    plt.title('Predizione (Futuro)')
    plt.legend()
    plt.show()

In [ ]:
lookback = 100
horizon = 10
batch_size = 2048
stride = 10
norm = "zscore"
norm_all = False
samples = (2500, 200, 200)
channel = 0
labels = True

In [ ]:
from lib.Dataloaders.VirtualAgentDataset import VirtualAgentDataset

# Percorso del tensore che abbiamo appena creato!
DATA_PATH = "../Preprocess/VirtualAgent/VA_Dataset_Tensor.npz"

print("Caricamento dataset in memoria...")
train_dataset = VirtualAgentDataset(DATA_PATH, split='train')
val_dataset = VirtualAgentDataset(DATA_PATH, split='val')
test_dataset = VirtualAgentDataset(DATA_PATH, split='test')

train = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# model = SimpleTransformerForecast(lookback=lookback,
#                                   horizon=horizon,
#                                   input_dim=8,
#                                   target_dim=1,
#                                   d_model=16,
#                                   n_heads=2,
#                                   num_layers=6, dropout=0).to('cuda')
# model.compile()

In [ ]:
# model = PatchTransformerForecast(lookback=lookback,
#                                 horizon=horizon,
#                                 input_dim=8,
#                                 target_dim=1,
#                                 d_model=64,
#                                 n_heads=8,
#                                 num_layers=2, dropout=0, patch_len=10).to('cuda')

In [ ]:
# model = PatchTransformerForecast2(lookback=lookback,
#                                   horizon=horizon,
#                                   input_dim=8,
#                                   target_dim=1,
#                                   d_model=64,
#                                   n_heads=8,
#                                   num_layers=2,
#                                   norm_first=True,
#                                   dropout=0,
#                                   patch_len=10,
#                                   layer_norm_eps=1e-5,
#                                   bias=True,
#                                   swiglu=True,
#                                   rmsnorm=True,
#                                   trans_norm=True,
#                                   device='cuda').to('cuda')
# model.compile()

In [ ]:
model = PatchTransformerExoForecast(
    lookback=lookback,
    horizon=horizon,
    input_dim=8,           # 8 distanze passate (Self + 7 Others)
    exo_dim=18,            # LE NOSTRE 18 FEATURE DELLA BARCA
    condition_dim=9,       # RowSide + Score (se il tuo script usa cond_dim scrivilo pure)
    target_dim=1,          # 1 distanza futura (Self)
    d_model=64,
    n_heads=8,
    num_layers=2,
    norm_first=True,
    dropout=0,
    patch_len=10,
    layer_norm_eps=1e-5,
    bias=True,
    swiglu=True,
    rmsnorm=True,
    trans_norm=True,
    device='cuda', 
    verbose=False
).to('cuda')
# model.compile()

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = StepLR(optimizer, step_size=20, gamma=0.90)

In [ ]:
# # Testing the model with random input
# random_input = torch.randn(32, 8, 120).to('cuda')
# model.to('cuda')
# model(random_input).shape

In [ ]:
hist_loss = train_loop(model, train, val, optimizer, scheduler, epochs=1000, patience=100)

In [ ]:
plt.plot(hist_loss['train'], label='train')
plt.plot(hist_loss['val'], label='val')
plt.legend()


In [ ]:
print(scheduler.get_last_lr())

In [ ]:
plot_prediction(model, train.dataset[0])

In [ ]:
plot_prediction(model, val.dataset[0])

In [ ]:
plot_prediction(model, test.dataset[0])

In [ ]:
def dataset_loss(dataset):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    data_loss = 0.0
    for i, (x_endo, x_exo, x_cond, y_target) in enumerate(dataset):
        # Sposta su GPU
        x_endo = x_endo.to(device)
        x_exo = x_exo.to(device)
        x_cond = x_cond.to(device)
        y = y_target.to(device)
        
        # Predizione e calcolo errore
        outputs = model(x_endo, exo=x_exo, cond=x_cond).squeeze(-1)
        loss = F.mse_loss(y.squeeze(-1), outputs)
        data_loss += loss.item()
        
    return data_loss / (i + 1)

print('Train loss:', dataset_loss(train))
print('Val loss:', dataset_loss(val))
print('Test loss:', dataset_loss(test))

In [ ]:
# Dopo che l'addestramento è finito e sei soddisfatto
percorso_salvataggio = 'weights_transformer_patch.pth'

# Salviamo SOLO lo state_dict (i pesi)
torch.save(model.state_dict(), percorso_salvataggio)
print(f"[*] Pesi salvati correttamente in: {percorso_salvataggio}")

In [ ]:
# 3. Generazione della tabella passandogli il modello e la forma esatta dell'input
# La forma dell'input per il PatchTransformer deve essere (Batch, Canali, Tempo) -> (1, 8, 100)
from torchinfo import summary

# (Batch, Canali, Tempo)
summary(model, input_data=(
    torch.randn(1, 8, lookback).cuda(),      # x_endo 
),
    exo=torch.randn(1, 18, lookback).cuda(), # exo 
    
    # ---> MODIFICATO QUI: Non ha più il tempo (lookback), è un token statico <---
    cond=torch.randn(1, 9).cuda(),           # cond (Batch, Features statiche)
    
    device='cuda',
    depth=4,
    mode="eval"
)